Docs:<br>
- [ReportLab Docs](https://docs.reportlab.com/reportlab/userguide/ch1_intro/)
- [StreamLit Gallery - voor ophalen van data](https://streamlit.io/gallery)

<br>

Basisdingen:
[How to iterate over all or certain columns of a df](https://www.geeksforgeeks.org/python/loop-or-iterate-over-all-or-certain-columns-of-a-dataframe-in-python-pandas/) 
<br>
Om naar te kijken: <br>
-  [Google resultaten concepts](https://www.google.com/search?q=app+pc+for+brainstorming+with+drawing+tablet&num=10&sca_esv=9151e0e90600ee3c&sxsrf=ANbL-n7LjWhKl-hEHRYIRnvv6TYRhGIzTA%3A1774340841409&ei=6UrCaenXGMqLi-gPo-_UgAM&biw=1712&bih=1326&ved=0ahUKEwip8L_cjriTAxXKxQIHHaM3FTAQ4dUDCBE&uact=5&oq=app+pc+for+brainstorming+with+drawing+tablet&gs_lp=Egxnd3Mtd2l6LXNlcnAiLGFwcCBwYyBmb3IgYnJhaW5zdG9ybWluZyB3aXRoIGRyYXdpbmcgdGFibGV0MgUQIRigATIFECEYoAEyBRAhGKABSMY-UABY1j1wBngBkAEAmAFwoAGwHaoBBDQ5LjG4AQPIAQD4AQGYAjigArUfwgILEAAYgAQYkQIYigXCAgoQABiABBhDGIoFwgIQEC4YgAQY0QMYQxjHARiKBcICBRAAGIAEwgILEC4YgAQY0QMYxwHCAgUQLhiABMICBhAAGBYYHsICBxAAGIAEGA3CAgYQABgNGB7CAgsQABiABBiGAxiKBcICBRAAGO8FwgIIEAAYgAQYogTCAgcQIRigARgKwgIFECEYnwXCAgQQIRgVmAMAkgcENTQuMqAH3JsCsgcENDguMrgHkh_CBwkwLjI5LjI2LjHIB6EBgAgA&sclient=gws-wiz-serp)
<br>

Aantekeningen:
<br>

Hoe zorg ik dat ik meerdere inputs df's kan verwerken in één uiteindelijke score?


In [8]:
import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from fpdf import FPDF
import tempfile
from pathlib import Path
import os
import glob

In [9]:
#Tijdelijke oplossing totdat ik andere manier heb gevonden om input te krijgen
folder_path = r'c:\Users\hans_\Documents\GitHub\Stakeholder_analysis\Testdingen'
pattern = os.path.join(folder_path, '*.csv')
csv_files = glob.glob(pattern)


print(csv_files)

[]


In [10]:
#Create function to synthesize assesment input
def synthesize_assessment_input(folder_name='data'):
    #Creëer een pad naar de map waar de csv-bestanden zich bevinden en zoek naar alle csv-bestanden in die map
    base_path = Path.cwd()/folder_name #Path.cwd() geeft de huidige werkmap terug. Base_path is die map plus de mapnaam
    pattern = '*.csv'
    csv_files = list(base_path.glob(pattern))

    if not csv_files: #als er geen csv-bestanden zijn gevonden, geef een foutmelding
        raise FileNotFoundError(f"No CSV files found in the folder: {base_path}")

    all_data = []

    
    #Doorloop alle csv bestanden en voeg ze samen in een dataframe
    for file in csv_files:
        temp_df = pd.read_csv(file, sep=';')
        all_data.append(temp_df)

    #Combineer alle dataframes in één dataframe
    combined_df = pd.concat(all_data, ignore_index=True)

    #Logica om de gecombineerde dataframe te verwerken en te synthetiseren
    aggregated_logic = {
        'formeel': 'mean',
        'informatie': 'mean',
        'informeel': 'mean',
        'legitimiteit': 'mean',
        'betrokkenheid': 'mean',
        'waarom': lambda x: ' | '.join(set(x))
    }

    synthesis = combined_df.groupby('stakeholder').agg(aggregated_logic).reset_index()
    #Mogelijk later toevoegen om af te ronden
    return synthesis

In [11]:
df = synthesize_assessment_input()
print(df.head())

     stakeholder   formeel  informatie  informeel  legitimiteit  \
0            FNV  4.000000    4.333333   4.333333      4.000000   
1  Stakeholder_d  5.000000    3.333333   3.666667      5.000000   
2            VNG  2.000000    2.666667   3.000000      3.666667   
3        VNO-NCW  4.333333    1.333333   3.000000      4.000000   

   betrokkenheid                            waarom  
0       3.333333  Heeft veel officiële bevoegdheid  
1       3.000000                 reden 4 | reden 3  
2       3.333333       Reden 4 | Reden 2 | Reden 3  
3       3.000000  Heeft geen officiële bevoegdheid  


In [12]:
#Check for Apple/WINDOWS path issues
print('cwd=', os.getcwd())
print(os.path.exists(r'c:\Users\hans_\Documents\GitHub\Stakeholder_analysis\files\.ipynb_checkpoints\input_stakeholders-checkpoint.csv'))


cwd= c:\Users\hans_\Documents\GitHub\Stakeholder_analysis\files
True


In [ ]:
#Check voor path_issues met MacOs/Windows
print(Path.cwd())

c:\Users\hans_\Documents\GitHub\Stakeholder_analysis\files


In [ ]:
#Outdated function to load data, replaced by synthesize_assessment_input
def load_data(file_path):
    try:
        return pd.read_csv(file_path, sep=';')
    except FileNotFoundError:
        return pd.DataFrame({"StakeHolder": ['Project']}) #temp 



In [15]:
#Structureren van data:
def data_structure(df):
    columns = df.columns.tolist()
    for col in columns:
        try:
            all_counts = df[col].value_counts()
            return all_counts
        except ValueError:
            print('wrong values')

In [16]:
def get_strategy(power, interest): #nodig: power en interest score op basis van input)
    if power >= 4 and interest >= 4: return "Manage closely"
    if power >= 4 and interest < 4: return "keep satisfied"
    if power < 4 and interest >= 4: return "keep informed"
    return "Monitor only"


In [ ]:
def create_power_legitimacy_urgency_columns(dataFrame):
    power_columns = ['formeel', 'informatie', 'informeel','legitimiteit']
    urgency_columns = 
    dataFrame['power_scores'] = dataFrame[power_columns].mean(axis=1)
    dataFrame['interest'] = dataFrame['betrokkenheid']

In [18]:
def create_matrix_plot(df):
    fig, ax = plt.subplots(figsize=(6,4))
    ax.scatter(df['power_scores'], df['interest'], c='blue')

    #Kwadranten indelen
    plt.axhline(3, color='black', linewidth=1)
    plt.axvline(3, color='black', linewidth=1)
    plt.xlim(1,5)
    plt.ylim(1,5)

    plt.xlabel('power (1-5)')
    plt.ylabel('interest (1-5)')
    plt.title('Stakeholder map')

    for i, txt in enumerate(df['stakeholder']):
        ax.annotate(txt, (df['power_scores'].iat[i], df['interest'].iat[i])) #.iat werkt als iloc, maar dan voor specifieke cellen, niet hele rijen of kolommen

    plt.tight_layout()
    plt.show()
    plot_path = tempfile.NamedTemporaryFile(delete=False, suffix=".png").name
    plt.savefig(plot_path)
    print(f"File location: {plot_path}")
    return plot_path

In [ ]:
#Functie voor het creëren van een venn-diagram van stakeholders.
# We creëren een plot met drie cirkels. Elk van de cirkels is één van de drie categorieën van stakeholders.
def create_venn_diagram(df):


In [19]:
create_power_interest_columns(df)
df['strategy'] = df.apply(lambda row: get_strategy(row['power_scores'], row['interest']), axis=1)


for i, row in df.iterrows():
    print(f"Stakeholder: {row['stakeholder']}, power: {row['power_scores']}, interest: {row['interest']}, Strategy: {row['strategy']}")


Stakeholder: FNV, power: 4.166666666666666, interest: 3.3333333333333335, Strategy: keep satisfied
Stakeholder: Stakeholder_d, power: 4.25, interest: 3.0, Strategy: keep satisfied
Stakeholder: VNG, power: 2.833333333333333, interest: 3.3333333333333335, Strategy: Monitor only
Stakeholder: VNO-NCW, power: 3.1666666666666665, interest: 3.0, Strategy: Monitor only


In [ ]:
#Plot stakeholders op kaart
create_matrix_plot(df)


File location: C:\Users\hans_\AppData\Local\Temp\tmp6czzybqn.png


C:\Users\hans_\AppData\Local\Temp\ipykernel_21572\1052847246.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


'C:\\Users\\hans_\\AppData\\Local\\Temp\\tmp6czzybqn.png'